In [2]:
import torch

### We can also check invidual functions 
https://discuss.pytorch.org/t/how-to-know-if-specific-function-is-differential-or-not/125960

In [ ]:
x = torch.randn((10, 10), requires_grad=True)
out = torch.where(x > 0.5, 1, 0)
print(out.grad_fn) # !!! NOT DIFFERENTIABLE 

None


In [6]:
x = torch.randn((10, 10), requires_grad=True)
out = torch.argwhere(x > 0.5)
print(out.grad_fn) # !!! NOT DIFFERENTIABLE 

None


Both of these functions are used in the https://github.com/junzis/contrail-seg/blob/main/loss.py implemtation by the paper: https://ieeexplore.ieee.org/stamp/stamp.jsp?arnumber=10820969. There is serious doubt on the differentibility of this loss function. It seems to me that the training procedure just treats this loss as a constant and there is no change in the loss (apart from parameters that are changed by the aux dice loss). This is also veriable by looking at their qualitative images which turn out to be quite similar to the dice ones. 

In [ ]:
import torch
import torch.nn as nn
import math
import segmentation_models_pytorch as smp

# --- YOUR ORIGINAL CLASS (with slight fixes for standalone running) ---
class HoughSRLoss(nn.Module):
    def __init__(self, alpha=0.5, num_theta=180, rho_bins=512, line_thresh=50):
        super().__init__()
        self.alpha = alpha
        self.num_theta = num_theta
        self.rho_bins = rho_bins
        self.line_thresh = line_thresh
        # Using a placeholder if SMP isn't installed, otherwise uses SMP
        self.dice_loss = smp.losses.DiceLoss(mode="binary", from_logits=False)

    def single_hough_map(self, mask2d: torch.Tensor) -> torch.Tensor:
        device = mask2d.device
        H, W = mask2d.shape

        # POINT OF FAILURE 1: Thresholding creates a non-differentiable mask
        binary = (mask2d > 0.5).float()

        # POINT OF FAILURE 2: torch.where(indices) is NOT differentiable
        ys, xs = torch.where(binary > 0)
        
        if xs.numel() == 0:
            return torch.zeros((self.rho_bins, self.num_theta), device=device)

        thetas = torch.linspace(-math.pi / 2, math.pi / 2, self.num_theta, device=device)
        cos_t = torch.cos(thetas)
        sin_t = torch.sin(thetas)

        diag = math.sqrt(H * H + W * W)
        rho_min, rho_max = -diag, diag
        acc = torch.zeros((self.rho_bins, self.num_theta), device=device)

        xs = xs.float()
        ys = ys.float()

        for t_idx in range(self.num_theta):
            rho_vals = xs * cos_t[t_idx] + ys * sin_t[t_idx]
            
            # POINT OF FAILURE 3: .long() casting kills gradients
            rho_idx = ((rho_vals - rho_min) / (rho_max - rho_min) * (self.rho_bins - 1)).long()
            rho_idx = torch.clamp(rho_idx, 0, self.rho_bins - 1)

            # POINT OF FAILURE 4: bincount is a discrete counting op
            counts = torch.bincount(rho_idx, minlength=self.rho_bins).float()
            acc[:, t_idx] = counts

        # POINT OF FAILURE 5: Hard thresholding in the accumulator
        acc = torch.where(acc >= self.line_thresh, acc, torch.zeros_like(acc))

        if acc.max() > 0:
            acc = acc / acc.max()

        return acc

    def forward(self, y_pred, y_true):
        # Image space Dice
        loss_dice = self.dice_loss(y_pred, y_true)
        
        # Hough space loss (Simplified for testing)
        pred_hough = self.batch_hough_maps(y_pred)
        true_hough = self.batch_hough_maps(y_true)
        
        # Dice between Hough maps
        loss_hough = self.dice_loss(pred_hough, true_hough)
        
        return (1 - self.alpha) * loss_dice + self.alpha * loss_hough

    def batch_hough_maps(self, x: torch.Tensor) -> torch.Tensor:
        maps = []
        for i in range(x.shape[0]):
            maps.append(self.single_hough_map(x[i, 0]))
        return torch.stack(maps, dim=0).unsqueeze(1)

# --- THE TESTER ---
def run_gradient_test():
    print(" Starting Differentiability Test...\n")
    
    # 1. Setup Input (B=1, C=1, H=32, W=32)
    # We use small sizes so the loop finishes quickly
    # requires_grad=True simulates the output of a model
    y_pred = torch.sigmoid(torch.randn((1, 1, 32, 32), requires_grad=True))
    y_true = torch.randint(0, 2, (1, 1, 32, 32)).float()
    
    criterion = HoughSRLoss(alpha=0.5, num_theta=30, rho_bins=64) # Small bins for speed

    # 2. Forward Pass
    try:
        loss = criterion(y_pred, y_true)
        print(f" Forward Pass successful. Loss value: {loss.item():.4f}")
    except Exception as e:
        print(f" Forward Pass failed! Error: {e}")
        return

    # 3. Backward Pass (The critical part)
    print(" Attempting Backward Pass...")
    try:
        loss.backward()
        
        # 4. Check results
        if y_pred.grad is None:
            print("\n RESULT: NON-DIFFERENTIABLE")
            print("The gradient is None. Your model's weights will NOT update using this loss.")
        elif torch.all(y_pred.grad == 0):
            print("\n RESULT: ZERO GRADIENTS")
            print("Gradients exist but are all zero. The chain is broken by discrete operations.")
        else:
            print("\n RESULT: DIFFERENTIABLE")
            print(f"Gradient Mean: {y_pred.grad.abs().mean().item():.8f}")
            
    except Exception as e:
        print(f"\n Backward Pass crashed! Error: {e}")

if __name__ == "__main__":
    run_gradient_test()

In [ ]:

def hough_transform(batch, threshold=50, return_coordinates=False):
    # Check 0: The starting point
    print(f"--- Gradient Trace Start ---")
    print(f"Input grad_fn: {batch.grad_fn}") 

    height, width = batch[0].squeeze().shape
    device = batch.device

    thetas = torch.arange(0, 180, 0.5, device=device)
    d = torch.sqrt(torch.tensor(width, device=device).float() ** 2 + 
                   torch.tensor(height, device=device).float() ** 2)
    rhos = torch.arange(-d, d, 3, device=device)

    cos_thetas = torch.cos(torch.deg2rad(thetas))
    sin_thetas = torch.sin(torch.deg2rad(thetas))

    hough_matrices = torch.zeros(
        batch.shape[0], rhos.shape[0] - 1, thetas.shape[0] - 1, device=device
    )

    for i, img in enumerate(batch):
        img = img.squeeze()
        
        # Check 1: Did squeezing break it?
        # (It shouldn't, but let's be sure)
        
        # KILLER 1: argwhere + comparison
        # Logic: img > 0.5 creates a Bool tensor (No gradient)
        # argwhere creates an Integer tensor (No gradient)
        mask = img > 0.5
        points = torch.argwhere(mask).type_as(cos_thetas)
        print(f"Step 1 (Points) grad_fn: {points.grad_fn}  <-- If None, chain broke at argwhere")

        # Check 2: Matrix Multiplication
        rho_values = torch.matmul(points, torch.stack((sin_thetas, cos_thetas)))
        print(f"Step 2 (Rho Values) grad_fn: {rho_values.grad_fn}")

        # KILLER 2: histogramdd
        # Histograms are counting operations. Moving a point 0.001mm 
        # doesn't change the bin count, so the derivative is 0.
        accumulator, _ = torch.histogramdd(
            torch.stack(
                (
                    torch.tile(thetas, (rho_values.shape[0],)),
                    rho_values.ravel(),
                )
            ).T,
            bins=[thetas, rhos],
        )
        print(f"Step 3 (Accumulator) grad_fn: {accumulator.grad_fn} <-- If None, chain broke at histogram")

        accumulator = torch.transpose(accumulator, 0, 1)

        if return_coordinates:
            hough_lines = torch.argwhere(accumulator > threshold)
            return hough_lines # This will never have a grad_fn

        # KILLER 3: Thresholding
        # torch.where with constants 1 and 0 is a step function.
        hough_matrix = torch.where(accumulator > threshold, 1.0, 0.0)
        print(f"Step 4 (Final Matrix) grad_fn: {hough_matrix.grad_fn} <-- If None, chain broke at threshold")
        
        hough_matrices[i] = hough_matrix

    return hough_matrices

# TEST RUN
input_batch = torch.randn((1, 1, 32, 32), requires_grad=True)
output = hough_transform(input_batch, threshold=10)

print("\n--- Final Result ---")
print(input_batch.grad)
if output.grad_fn is not None:
    print("SUCCESS: Output is connected to Input.")
else:
    print("FAILURE: Gradient chain is broken.")

In [ ]:
import torch 

def hough_transform(batch, threshold=50, return_coordinates=False):
    # Check 0: The starting point
    print(f"--- Gradient Trace Start ---")
    print(f"Input grad_fn: {batch.grad_fn}") 

    height, width = batch[0].squeeze().shape
    device = batch.device

    thetas = torch.arange(0, 180, 0.5, device=device)
    d = torch.sqrt(torch.tensor(width, device=device).float() ** 2 + 
                   torch.tensor(height, device=device).float() ** 2)
    rhos = torch.arange(-d, d, 3, device=device)

    cos_thetas = torch.cos(torch.deg2rad(thetas))
    sin_thetas = torch.sin(torch.deg2rad(thetas))

    hough_matrices = torch.zeros(
        batch.shape[0], rhos.shape[0] - 1, thetas.shape[0] - 1, device=device
    )

    for i, img in enumerate(batch):
        img = img.squeeze()
        
        # Check 1: Did squeezing break it?
        # (It shouldn't, but let's be sure)
        
        # KILLER 1: argwhere + comparison
        # Logic: img > 0.5 creates a Bool tensor (No gradient)
        # argwhere creates an Integer tensor (No gradient)
        mask = img > 0.5
        points = torch.argwhere(mask).type_as(cos_thetas)
        print(f"Step 1 (Points) grad_fn: {points.grad_fn}  <-- If None, chain broke at argwhere")

        # Check 2: Matrix Multiplication
        rho_values = torch.matmul(points, torch.stack((sin_thetas, cos_thetas)))
        print(f"Step 2 (Rho Values) grad_fn: {rho_values.grad_fn}")

        # KILLER 2: histogramdd
        # Histograms are counting operations. Moving a point 0.001mm 
        # doesn't change the bin count, so the derivative is 0.
        accumulator, _ = torch.histogramdd(
            torch.stack(
                (
                    torch.tile(thetas, (rho_values.shape[0],)),
                    rho_values.ravel(),
                )
            ).T,
            bins=[thetas, rhos],
        )
        print(f"Step 3 (Accumulator) grad_fn: {accumulator.grad_fn} <-- If None, chain broke at histogram")

        accumulator = torch.transpose(accumulator, 0, 1)

        if return_coordinates:
            hough_lines = torch.argwhere(accumulator > threshold)
            return hough_lines # This will never have a grad_fn

        # KILLER 3: Thresholding
        # torch.where with constants 1 and 0 is a step function.
        hough_matrix = torch.where(accumulator > threshold, 1.0, 0.0)
        print(f"Step 4 (Final Matrix) grad_fn: {hough_matrix.grad_fn} <-- If None, chain broke at threshold")
        
        hough_matrices[i] = hough_matrix

    return hough_matrices

# TEST RUN
input_batch = torch.randn((1, 1, 32, 32), requires_grad=True)
output = hough_transform(input_batch, threshold=10)

print("\n--- Final Result ---")
print(input_batch.grad)
if output.grad_fn is not None:
    print("SUCCESS: Output is connected to Input.")
else:
    print("FAILURE: Gradient chain is broken.")